In [3]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY) 

# (Commented out) IPython autoreload extension for Jupyter notebook development
# %load_ext autoreload
# %autoreload 2
# %reload_ext autoreload 

# Import custom modules
from backend.extract_filter import classify_and_extract
from backend.data_utils import fetch_papers, verify_results, chunk_papers, format_query
from backend.Database import create_index, save_embeddings_to_index, delete_index
from backend.retrieval import answer_query, clear_memory, retrieve_chunks

from pinecone import Pinecone
import io
import contextlib

# Create and manage vector index
import pandas as pd
from urllib.error import HTTPError

# Importing async and evaluation tools
import asyncio
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import SemanticSimilarity
from ragas.embeddings import LangchainEmbeddingsWrapper

# Initialize ClearML for experiment tracking and artifact logging
import os
from dotenv import load_dotenv

load_dotenv()

CLEARML_WEB_HOST = os.getenv("CLEARML_WEB_HOST")
CLEARML_API_HOST = os.getenv("CLEARML_API_HOST")
CLEARML_FILES_HOST = os.getenv("CLEARML_FILES_HOST")
CLEARML_API_ACCESS_KEY = os.getenv("CLEARML_API_ACCESS_KEY")
CLEARML_API_SECRET_KEY = os.getenv("CLEARML_API_SECRET_KEY")

from clearml import Task, Logger
from time import sleep

'export' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


Pinecone API Key: [REDACTED]


In [4]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

Connect to ClearML

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

CLEARML_WEB_HOST = os.getenv("CLEARML_WEB_HOST")
CLEARML_API_HOST = os.getenv("CLEARML_API_HOST")
CLEARML_FILES_HOST = os.getenv("CLEARML_FILES_HOST")
CLEARML_API_ACCESS_KEY = os.getenv("CLEARML_API_ACCESS_KEY")
CLEARML_API_SECRET_KEY = os.getenv("CLEARML_API_SECRET_KEY")

from clearml import Task, Logger
task = Task.init(
    project_name="MyProject", 
    task_name="Embedding Comparison - inegrated env"
)

env: CLEARML_WEB_HOST=https://app.5ccsagap.er.kcl.ac.uk/
env: CLEARML_API_HOST=https://api.5ccsagap.er.kcl.ac.uk
env: CLEARML_FILES_HOST=https://files.5ccsagap.er.kcl.ac.uk
env: CLEARML_API_ACCESS_KEY=[REDACTED]
env: CLEARML_API_SECRET_KEY=[REDACTED]
ClearML Task: overwriting (reusing) task id=f685111c607f43f08b390b3bdf042e55
ClearML results page: https://app.5ccsagap.er.kcl.ac.uk/projects/f6c6e15540994198a771be0667091232/experiments/f685111c607f43f08b390b3bdf042e55/output/log


delete pinecone indexes exisited

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

from pinecone import Pinecone

indexes = pc.list_indexes()

# delete all indexes
for idx in indexes:
    index_name = idx["name"]
    pc.delete_index(index_name)
    print(f"Deleted index: {index_name}")



Define a new function that performs retrieval only

In [7]:
def retrieve_ans(user_input, index_obj): 
    classification = classify_and_extract(user_input)
    print(classification)
    
 
    filters = classification['filters']
    print("\nRunning Search with filters:", filters)

    # check if academic query is present
    if not format_query(filters):   
        question = classification['question']
        answer = answer_query(question, index_obj, top_k=100)
        print("Generated Answer:")
        print(answer)
        return

    # Step 1: Fetch papers
    papers = fetch_papers(filters)

    # Step 2: Verify & Filter papers
    verified_papers = verify_results(papers, filters)
    print(f"[integration] Found {len(verified_papers)} verified paper(s).")
    
    # Determine number of papers to return; default to 1 if not provided
    try:
        num_papers = int(filters.get("max_results", 1))
    except ValueError:
        num_papers = 1
    num_papers = min(num_papers, len(verified_papers))

    # Step 3: Chunk papers
    chunks = chunk_papers(verified_papers[:num_papers])  # Get full paper content as chunks
    texts = [chunk["text"] for chunk in chunks]  # Extract just the text

    if texts:
        # Save the full paper text chunks to the index
        batch_size = 200
        for i in range(0, len(texts), batch_size):
            text_batch = texts[i:i+batch_size] 
            save_embeddings_to_index(index_name=index_name, text=text_batch)
        sleep(1)

    ans = retrieve_chunks(user_input, index_obj)

    return ans


Load self-constructured benchmark

In [8]:
import pandas as pd
filename = "evaluation_data/question_answer.csv"
df = pd.read_csv(filename, encoding='ISO-8859-1')
print(df.head(0))

Empty DataFrame
Columns: [question, answer, paper, reference paragraph 1, reference paragraph 2, reference paragraph 3, similar_question]
Index: []


# Evaluate text-embedding-3-small

In [9]:
from dotenv import load_dotenv
import os 
load_dotenv()
os.environ["EMBEDDING_MODEL"] = "text-embedding-3-small"
os.environ["USE_OPENAI_EMBEDDINGS"] = 'True'

model_name = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
print("Using embedding model:", model_name)

model_name = os.getenv("USE_OPENAI_EMBEDDINGS", 'True')
print("whether use openai embedding model:", model_name)
# sometimes you need to restart the kernel in order for this env settings to work

Using embedding model: text-embedding-3-small
whether use openai embedding model: True


In [10]:
import sys
sys.setrecursionlimit(10000)


In [15]:
index_name = create_index()
index_obj = pc.Index(index_name)

Index 'g1ze6wzztb' created successfully!


In [16]:
import pandas as pd
import numpy as np
import builtins 
from backend.Embedder import embed
from sentence_transformers import SentenceTransformer, util

# Load pre-trained SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Assumes df is already loaded and contains the following columns:
# 'question', 'reference paragraph 1', 'reference paragraph 2', 'reference paragraph 3'
# Example: df = pd.read_csv("your_dataset.csv")

all_scores = []         # Stores cosine similarity scores for each question (3 scores per question)
comparison_results = [] # Stores detailed comparison results to be converted into a DataFrame

# Iterate over each row (question) in the DataFrame
for idx, row in df.iterrows():
    print("=" * 60)
    print(f"Question {idx+1}:")
    print(row['question'])
    print("=" * 60)
    
    # Extract the user query (question)
    user_query = row['question']
    
    # Reference paragraphs to compare against, ordered as expected
    ref_paragraphs = [
        row['reference paragraph 1'],
        row['reference paragraph 2'],
        row['reference paragraph 3']
    ]
    
    # Retrieve top-k results using the retrieval function
    # ans should be a list of dicts with "text" and "source"
    try:
        ans = retrieve_ans(user_query, index_obj)
    except KeyError as e:
        print(f"KeyError encountered for question {idx+1}: {e}. Skipping this question.\n")
        continue

    # Skip if there are not enough results to compare
    if ans == 0 or len(ans) < 3:
        print(f"WARNING: No sufficient results for question {idx+1}: {user_query}\n")
        continue
    
    scores = []
    # Compare the top 3 retrieved results with the corresponding reference paragraphs
    for i in range(3):
        result_text = ans[i]['text']
        ref_text = ref_paragraphs[i]
        
        print(f"\n--- Comparison {i+1} ---")
        print("Retrieved Result:")
        print(result_text)
        print("\nReference Paragraph:")
        print(ref_text)
        
        # Compute sentence embeddings
        emb_result = model.encode(result_text, convert_to_tensor=True)
        emb_ref = model.encode(ref_text, convert_to_tensor=True)
        
        # Compute cosine similarity (util.cos_sim returns a tensor)
        score = util.cos_sim(emb_result, emb_ref)
        similarity = score.item()
        scores.append(similarity)
        
        print(f"\nCosine Similarity for Comparison {i+1}: {similarity:.4f}")
        
        # Save current comparison to the result list
        comparison_results.append({
            "question": user_query,
            "comparison_index": i+1,
            "retrieved_result": result_text,
            "reference_paragraph": ref_text,
            "cosine_similarity": similarity
        })
    
    print("\nSimilarity scores for Question {}: {}".format(idx+1, scores))
    all_scores.append(scores)

# Convert scores to a NumPy array: shape = (num_questions, 3)
score_matrix = np.array(all_scores)
print("\n" + "=" * 60)
print("Cosine Similarity Matrix:")
print(score_matrix)

# Compute overall mean cosine similarity
mean_score = score_matrix.mean()
print("\nMean Cosine Similarity: {:.4f}".format(mean_score))

# Convert all comparison results to a DataFrame
comparison_df = pd.DataFrame(comparison_results)
print("\nComparison Results DataFrame:")
print(comparison_df)


Question 1:
Describe the K-means clustering algorithm.
{'filters': {'query': 'K-means clustering algorithm', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'relevance'}, 'question': 'Describe the K-means clustering algorithm.'}

Running Search with filters: {'query': 'K-means clustering algorithm', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'relevance'}
[integration] Found 1 verified paper(s).
Extracted 89 chunks from 'An implementation of the relational k-means algorithm'

Total Chunks Extracted: 89

Upserted 89 embeddings into index 'g1ze6wzztb', namespace 'default'.
⚠️ No relevant documents found in Pinecone, wait for 5 seconds and trying again...
After retrying, relevant documents are found


In [ ]:
comparison_df

In [ ]:
logger = task.get_logger()
 
logger.report_table(
    title="Comparison Results - text-embedding-3-small",          
    series="Retrieval Evaluation",        
    iteration=0,                          
    table_plot=comparison_df              
)

In [ ]:
# Calculate the average cosine similarity from the table
avg_cosine_similarity = comparison_df["cosine_similarity"].mean()

# Log the average cosine similarity as a scalar metric
logger.report_scalar(
    title="Average Cosine Similarity - text-embedding-3-small",                       
    series="Retrieval Evaluation - Retrieval Cosine Similarity",                    
    value=avg_cosine_similarity,                            
    iteration=0                                              
)

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import util

# Assumes df is already loaded and contains the following columns:
# 'question', 'similar_question', 'reference paragraph 1', 'reference paragraph 2', 'reference paragraph 3'
# Also assumes comparison_df is previously generated and contains:
# "question", "comparison_index", "retrieved_result", "reference_paragraph", "cosine_similarity"

# List to store new similarity comparison results
new_comparison_results = []

# Iterate over the first 10 rows of df (adjust as needed)
for idx, row in df.head(10).iterrows():
    orig_question = row['question']
    sim_question = row['similar_question']
    
    print("=" * 60)
    print(f"Processing row {idx+1}:")
    print("Original Question:")
    print(orig_question)
    print("Similar Question:")
    print(sim_question)
    print("=" * 60)
    
    # Retrieve the original question's results from the previous comparison_df (ensure 1–3 order)
    orig_results_df = comparison_df[comparison_df['question'] == orig_question]
    if orig_results_df.empty or len(orig_results_df) < 3:
        print(f"WARNING: Not enough retrieval results for original question {idx+1}. Skipping.")
        continue
    
    orig_results = orig_results_df.sort_values("comparison_index")["retrieved_result"].tolist()
    
    # Retrieve results for the similar question using the retrieval function
    ans_sim = retrieve_ans(sim_question, index_obj)
    if ans_sim == 0 or len(ans_sim) < 3:
        print(f"WARNING: Not enough retrieval results for similar question {idx+1}. Skipping.")
        continue
    sim_results = [item['text'] for item in ans_sim[:3]]
    
    # Compute cosine similarity between the original and similar questions themselves
    emb_orig_question = model.encode(orig_question, convert_to_tensor=True)
    emb_sim_question = model.encode(sim_question, convert_to_tensor=True)
    query_cos_sim = util.cos_sim(emb_orig_question, emb_sim_question).item()
    
    # Compare the corresponding retrieval results at each position (1 vs 1, 2 vs 2, etc.)
    for i in range(3):
        orig_text = orig_results[i]
        sim_text = sim_results[i]
        
        emb_orig = model.encode(orig_text, convert_to_tensor=True)
        emb_sim = model.encode(sim_text, convert_to_tensor=True)
        retrieval_cos_sim = util.cos_sim(emb_orig, emb_sim).item()
        
        new_comparison_results.append({
            "original_question": orig_question,
            "similar_question": sim_question,
            "comparison_index": i+1,
            "orig_retrieved_result": orig_text,
            "sim_retrieved_result": sim_text,
            "retrieval_cosine_similarity": retrieval_cos_sim,
            "query_cosine_similarity": query_cos_sim
        })
        
        print(f"\nComparison {i+1}:")
        print("Original Retrieval Result:")
        print(orig_text)
        print("\nSimilar Retrieval Result:")
        print(sim_text)
        print(f"\nRetrieval Cosine Similarity: {retrieval_cos_sim:.4f}")
    
    print(f"\nQuery Cosine Similarity (Original vs. Similar Question): {query_cos_sim:.4f}")

# Convert results into a DataFrame
similarity_comparison_df = pd.DataFrame(new_comparison_results)
print("\nNew Comparison Results DataFrame:")
print(similarity_comparison_df)

# Compute the average similarity scores across all comparisons
mean_retrieval_cos_sim = similarity_comparison_df["retrieval_cosine_similarity"].mean()
mean_query_cos_sim = similarity_comparison_df["query_cosine_similarity"].mean()

print("\nMean Retrieval Cosine Similarity: {:.4f}".format(mean_retrieval_cos_sim))
print("Mean Query Cosine Similarity: {:.4f}".format(mean_query_cos_sim))


In [ ]:
similarity_comparison_df

In [ ]:
logger = task.get_logger()
 
logger.report_table(
    title="Comparison Results - text-embedding-3-small",         
    series="Embedding Evaluation",        
    iteration=0,                         
    table_plot=similarity_comparison_df           
)

In [ ]:
# Compute average cosine similarities
avg_retrieval_cosine = similarity_comparison_df["retrieval_cosine_similarity"].mean()
avg_query_cosine = similarity_comparison_df["query_cosine_similarity"].mean()

# Log scalar values
logger.report_scalar(
    title="Average Cosine Similarity - text-embedding-3-small",
    series="Embedding Evaluation - Retrieval Cosine Similarity",
    value=avg_retrieval_cosine,
    iteration=0
)

logger.report_scalar(
    title="Average Cosine Similarity - text-embedding-3-small",
    series="Embedding Evaluation - Query Cosine Similarity",
    value=avg_query_cosine,
    iteration=0
)

# Evaluate text-embedding-3-large

In [ ]:
index_name = create_index()
index_obj = pc.Index(index_name)

In [ ]:
from dotenv import load_dotenv
import os 
load_dotenv()
os.environ["EMBEDDING_MODEL"] = 'text-embedding-3-large'
os.environ["USE_OPENAI_EMBEDDINGS"] = 'True'

model_name = os.getenv("EMBEDDING_MODEL", 'text-embedding-3-large')
print("Using embedding model:", model_name) 

model_name = os.getenv("USE_OPENAI_EMBEDDINGS", 'True')
print("whether use openai embedding model:", model_name)
# sometimes you need to restart the kernel in order for this env settings to work

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

# Load the pre-trained SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Assumes df is already loaded and contains the following columns:
# 'question', 'reference paragraph 1', 'reference paragraph 2', 'reference paragraph 3'
# Example: df = pd.read_csv("your_dataset.csv")

all_scores = []         # Stores three cosine similarity scores per question
comparison_results = [] # Stores all comparison details for conversion to DataFrame

# Iterate through each row in the DataFrame (e.g., 10 questions = 10 rows = 30 comparisons)
for idx, row in df.iterrows():
    print("=" * 60)
    print(f"Question {idx+1}:")
    print(row['question'])
    print("=" * 60)
    
    # Extract the current question
    user_query = row['question']
    
    # Extract the reference paragraphs corresponding to the expected top-3 retrieved results
    ref_paragraphs = [
        row['reference paragraph 1'],
        row['reference paragraph 2'],
        row['reference paragraph 3']
    ]
    
    # Retrieve top results using the retrieval function
    # ans should be a list of dictionaries with keys: "text" and "source"
    try:
        ans = retrieve_ans(user_query, index_obj)
    except KeyError as e:
        print(f"KeyError encountered for question {idx+1}: {e}. Skipping this question.\n")
        continue

    if ans == 0 or len(ans) < 3:
        print(f"WARNING: Not enough results for question {idx+1}: {user_query}\n")
        continue
    
    scores = []
    # Compare the top 3 retrieved results against the 3 reference paragraphs in order
    for i in range(3):
        result_text = ans[i]['text']
        ref_text = ref_paragraphs[i]
        
        print(f"\n--- Comparison {i+1} ---")
        print("Retrieved Result:")
        print(result_text)
        print("\nReference Paragraph:")
        print(ref_text)
        
        # Generate embeddings for both texts
        emb_result = model.encode(result_text, convert_to_tensor=True)
        emb_ref = model.encode(ref_text, convert_to_tensor=True)
        
        # Compute cosine similarity (util.cos_sim returns a tensor)
        score = util.cos_sim(emb_result, emb_ref)
        similarity = score.item()
        scores.append(similarity)
        
        print(f"\nCosine Similarity for Comparison {i+1}: {similarity:.4f}")
        
        # Store the comparison result
        comparison_results.append({
            "question": user_query,
            "comparison_index": i+1,
            "retrieved_result": result_text,
            "reference_paragraph": ref_text,
            "cosine_similarity": similarity
        })
    
    print("\nSimilarity scores for Question {}: {}".format(idx+1, scores))
    all_scores.append(scores)

# Convert the list of scores into a NumPy array with shape (num_questions, 3)
score_matrix = np.array(all_scores)
print("\n" + "=" * 60)
print("Cosine Similarity Matrix:")
print(score_matrix)

# Compute the overall mean cosine similarity
mean_score = score_matrix.mean()
print("\nMean Cosine Similarity: {:.4f}".format(mean_score))

# Convert the list of detailed comparison results to a DataFrame and display it
comparison_df = pd.DataFrame(comparison_results)
print("\nComparison Results DataFrame:")
print(comparison_df)


In [ ]:
comparison_df

In [ ]:
logger = task.get_logger()
 
logger.report_table(
    title="Comparison Results - text-embedding-3-large",           
    series="Retrieval Evaluation",        
    iteration=0,                         
    table_plot=comparison_df             
)

In [ ]:
# Calculate the average cosine similarity from the table
avg_cosine_similarity = comparison_df["cosine_similarity"].mean()

# Log the average cosine similarity as a scalar metric
logger.report_scalar(
    title="Average Cosine Similarity - text-embedding-3-large",                       
    series="Retrieval Evaluation - Retrieval Cosine Similarity",                    
    value=avg_cosine_similarity,                            
    iteration=0                                              
)

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import util

# Assumes df is already loaded and contains the following columns:
# 'question', 'similar_question', 'reference paragraph 1', 'reference paragraph 2', 'reference paragraph 3'
# Also assumes comparison_df has been generated earlier and contains:
# "question", "comparison_index", "retrieved_result", "reference_paragraph", "cosine_similarity"

# List to store new comparison results
new_comparison_results = []

# Iterate over the first 10 rows of df (adjust the number as needed)
for idx, row in df.head(10).iterrows():
    orig_question = row['question']
    sim_question = row['similar_question']
    
    print("=" * 60)
    print(f"Processing row {idx+1}:")
    print("Original Question:")
    print(orig_question)
    print("Similar Question:")
    print(sim_question)
    print("=" * 60)
    
    # Retrieve the original question's retrieval results
    # Filter comparison_df to get results for the original question (ensure order 1–3)
    orig_results_df = comparison_df[comparison_df['question'] == orig_question]
    if orig_results_df.empty or len(orig_results_df) < 3:
        print(f"WARNING: Not enough retrieval results for original question {idx+1}. Skipping row.")
        continue
    
    orig_results = orig_results_df.sort_values("comparison_index")["retrieved_result"].tolist()
    
    # Retrieve results for the similar question using the retrieval function
    ans_sim = retrieve_ans(sim_question, index_obj)
    if ans_sim == 0 or len(ans_sim) < 3:
        print(f"WARNING: Not enough retrieval results for similar question {idx+1}. Skipping row.")
        continue
    sim_results = [item['text'] for item in ans_sim[:3]]
    
    # Compute cosine similarity between the original and similar questions themselves
    emb_orig_question = model.encode(orig_question, convert_to_tensor=True)
    emb_sim_question = model.encode(sim_question, convert_to_tensor=True)
    query_cos_sim = util.cos_sim(emb_orig_question, emb_sim_question).item()
    
    # Compare corresponding retrieved results: 1 vs 1, 2 vs 2, 3 vs 3
    for i in range(3):
        orig_text = orig_results[i]
        sim_text = sim_results[i]
        
        emb_orig = model.encode(orig_text, convert_to_tensor=True)
        emb_sim = model.encode(sim_text, convert_to_tensor=True)
        retrieval_cos_sim = util.cos_sim(emb_orig, emb_sim).item()
        
        new_comparison_results.append({
            "original_question": orig_question,
            "similar_question": sim_question,
            "comparison_index": i+1,
            "orig_retrieved_result": orig_text,
            "sim_retrieved_result": sim_text,
            "retrieval_cosine_similarity": retrieval_cos_sim,
            "query_cosine_similarity": query_cos_sim
        })
        
        print(f"\nComparison {i+1}:")
        print("Original Retrieval Result:")
        print(orig_text)
        print("\nSimilar Retrieval Result:")
        print(sim_text)
        print(f"\nRetrieval Cosine Similarity: {retrieval_cos_sim:.4f}")
    
    print(f"\nQuery Cosine Similarity (Original vs. Similar Question): {query_cos_sim:.4f}")

# Convert the new comparison results into a DataFrame
similarity_comparison_df = pd.DataFrame(new_comparison_results)
print("\nNew Comparison Results DataFrame:")
print(similarity_comparison_df)

# Compute mean cosine similarity across all comparisons
mean_retrieval_cos_sim = similarity_comparison_df["retrieval_cosine_similarity"].mean()
mean_query_cos_sim = similarity_comparison_df["query_cosine_similarity"].mean()

print("\nMean Retrieval Cosine Similarity: {:.4f}".format(mean_retrieval_cos_sim))
print("Mean Query Cosine Similarity: {:.4f}".format(mean_query_cos_sim))


In [ ]:
similarity_comparison_df

In [ ]:
logger = task.get_logger()
 
logger.report_table(
    title="Comparison Results - text-embedding-3-large",          
    series="Embedding Evaluation",        
    iteration=0,                          
    table_plot=similarity_comparison_df              
)

In [ ]:
# Compute average cosine similarities
avg_retrieval_cosine = similarity_comparison_df["retrieval_cosine_similarity"].mean()
avg_query_cosine = similarity_comparison_df["query_cosine_similarity"].mean()

# Log scalar values
logger.report_scalar(
    title="Average Cosine Similarity - text-embedding-3-large",
    series="Embedding Evaluation - Retrieval Cosine Similarity",
    value=avg_retrieval_cosine,
    iteration=0
)

logger.report_scalar(
    title="Average Cosine Similarity - text-embedding-3-large",
    series="Embedding Evaluation - Query Cosine Similarity",
    value=avg_query_cosine,
    iteration=0
)

# Evaluate all-MiniLM-L6-v2

In [ ]:
index_name = create_index()
index_obj = pc.Index(index_name)

In [ ]:
from dotenv import load_dotenv
import os 
load_dotenv() 
os.environ["USE_OPENAI_EMBEDDINGS"] = 'False'

model_name = os.getenv("USE_OPENAI_EMBEDDINGS", 'False')
print("whether use openai embedding model:", model_name)
# sometimes you need to restart the kernel in order for this env settings to work

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import util

# Assumes df is already loaded and contains the following columns:
# 'question', 'reference paragraph 1', 'reference paragraph 2', 'reference paragraph 3'
# Example: df = pd.read_csv("your_dataset.csv")

all_scores = []         # Stores three cosine similarity scores for each question
comparison_results = [] # Stores all comparison details to be converted into a DataFrame

# Iterate through each row in the DataFrame (e.g., 10 questions = 10 rows = 30 comparisons)
for idx, row in df.iterrows():
    print("=" * 60)
    print(f"Question {idx+1}:")
    print(row['question'])
    print("=" * 60)
    
    # Extract the user query from the current row
    user_query = row['question']
    
    # Prepare the list of reference paragraphs in the expected order
    ref_paragraphs = [
        row['reference paragraph 1'],
        row['reference paragraph 2'],
        row['reference paragraph 3']
    ]
    
    # Call the retrieval function for the current query
    # ans should be a list of dictionaries, each containing {"text": ..., "source": ...}
    try:
        ans = retrieve_ans(user_query, index_obj)
    except KeyError as e:
        print(f"KeyError encountered for question {idx+1}: {e}. Skipping this question.\n")
        continue

    if ans == 0 or len(ans) < 3:
        print(f"WARNING: Not enough results for question {idx+1}: {user_query}\n")
        continue
    
    scores = []
    # Compare the top 3 retrieved results against the 3 reference paragraphs in order
    for i in range(3):
        result_text = ans[i]['text']
        ref_text = ref_paragraphs[i]
        
        print(f"\n--- Comparison {i+1} ---")
        print("Retrieved Result:")
        print(result_text)
        print("\nReference Paragraph:")
        print(ref_text)
        
        # Generate sentence embeddings for both retrieved and reference texts
        emb_result = model.encode(result_text, convert_to_tensor=True)
        emb_ref = model.encode(ref_text, convert_to_tensor=True)
        
        # Compute cosine similarity (util.cos_sim returns a tensor)
        score = util.cos_sim(emb_result, emb_ref)
        similarity = score.item()
        scores.append(similarity)
        
        print(f"\nCosine Similarity for Comparison {i+1}: {similarity:.4f}")
        
        # Store the comparison result
        comparison_results.append({
            "question": user_query,
            "comparison_index": i+1,
            "retrieved_result": result_text,
            "reference_paragraph": ref_text,
            "cosine_similarity": similarity
        })
    
    print("\nSimilarity scores for Question {}: {}".format(idx+1, scores))
    all_scores.append(scores)

# Convert all scores into a NumPy array of shape (num_questions, 3)
score_matrix = np.array(all_scores)
print("\n" + "=" * 60)
print("Cosine Similarity Matrix:")
print(score_matrix)

# Calculate the overall mean cosine similarity
mean_score = score_matrix.mean()
print("\nMean Cosine Similarity: {:.4f}".format(mean_score))

# Convert the comparison results list into a DataFrame and display it
comparison_df = pd.DataFrame(comparison_results)
print("\nComparison Results DataFrame:")
print(comparison_df)


In [ ]:
comparison_df

In [ ]:
logger = task.get_logger()
 
logger.report_table(
    title="Comparison Results - all-MiniLM-L6-v2",          
    series="Retrieval Evaluation",       
    iteration=0,                          
    table_plot=comparison_df            
)

In [ ]:
# Calculate the average cosine similarity from the table
avg_cosine_similarity = comparison_df["cosine_similarity"].mean()

# Log the average cosine similarity as a scalar metric
logger.report_scalar(
    title="Average Cosine Similarity - all-MiniLM-L6-v2",                       
    series="Retrieval Evaluation - Retrieval Cosine Similarity",                    
    value=avg_cosine_similarity,                            
    iteration=0                                              
)

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import util

# Assumes df is already loaded and contains the following columns:
# 'question', 'similar_question', 'reference paragraph 1', 'reference paragraph 2', 'reference paragraph 3'
# Also assumes that comparison_df is already generated and contains:
# "question", "comparison_index", "retrieved_result", "reference_paragraph", "cosine_similarity"

# List to store new comparison results
new_comparison_results = []

# Iterate through the first 10 rows of df (adjust as needed)
for idx, row in df.head(10).iterrows():
    orig_question = row['question']
    sim_question = row['similar_question']
    
    print("=" * 60)
    print(f"Processing row {idx+1}:")
    print("Original Question:")
    print(orig_question)
    print("Similar Question:")
    print(sim_question)
    print("=" * 60)
    
    # Retrieve results for the original question from comparison_df (ensure order by index)
    orig_results_df = comparison_df[comparison_df['question'] == orig_question]
    if orig_results_df.empty or len(orig_results_df) < 3:
        print(f"WARNING: Not enough retrieval results for original question {idx+1}. Skipping row.")
        continue
    
    orig_results = orig_results_df.sort_values("comparison_index")["retrieved_result"].tolist()
    
    # Retrieve top-3 results for the similar question using the retrieval function
    ans_sim = retrieve_ans(sim_question, index_obj)
    if ans_sim == 0 or len(ans_sim) < 3:
        print(f"WARNING: Not enough retrieval results for similar question {idx+1}. Skipping row.")
        continue
    sim_results = [item['text'] for item in ans_sim[:3]]
    
    # Compute cosine similarity between the original and similar questions themselves
    emb_orig_question = model.encode(orig_question, convert_to_tensor=True)
    emb_sim_question = model.encode(sim_question, convert_to_tensor=True)
    query_cos_sim = util.cos_sim(emb_orig_question, emb_sim_question).item()
    
    # Compare each retrieval result at the same position: 1 vs 1, 2 vs 2, 3 vs 3
    for i in range(3):
        orig_text = orig_results[i]
        sim_text = sim_results[i]
        
        emb_orig = model.encode(orig_text, convert_to_tensor=True)
        emb_sim = model.encode(sim_text, convert_to_tensor=True)
        retrieval_cos_sim = util.cos_sim(emb_orig, emb_sim).item()
        
        new_comparison_results.append({
            "original_question": orig_question,
            "similar_question": sim_question,
            "comparison_index": i+1,
            "orig_retrieved_result": orig_text,
            "sim_retrieved_result": sim_text,
            "retrieval_cosine_similarity": retrieval_cos_sim,
            "query_cosine_similarity": query_cos_sim
        })
        
        print(f"\nComparison {i+1}:")
        print("Original Retrieval Result:")
        print(orig_text)
        print("\nSimilar Retrieval Result:")
        print(sim_text)
        print(f"\nRetrieval Cosine Similarity: {retrieval_cos_sim:.4f}")
    
    print(f"\nQuery Cosine Similarity (Original vs. Similar Question): {query_cos_sim:.4f}")

# Convert the comparison results into a DataFrame
similarity_comparison_df = pd.DataFrame(new_comparison_results)
print("\nNew Comparison Results DataFrame:")
print(similarity_comparison_df)

# Compute the mean cosine similarities across all comparisons
mean_retrieval_cos_sim = similarity_comparison_df["retrieval_cosine_similarity"].mean()
mean_query_cos_sim = similarity_comparison_df["query_cosine_similarity"].mean()

print("\nMean Retrieval Cosine Similarity: {:.4f}".format(mean_retrieval_cos_sim))
print("Mean Query Cosine Similarity: {:.4f}".format(mean_query_cos_sim))


In [ ]:
similarity_comparison_df

In [ ]:
logger = task.get_logger()
 
logger.report_table(
    title="Comparison Results - all-MiniLM-L6-v2",           
    series="Embedding Evaluation",         
    iteration=0,                          
    table_plot=similarity_comparison_df             
)

In [ ]:
# Compute average cosine similarities
avg_retrieval_cosine = similarity_comparison_df["retrieval_cosine_similarity"].mean()
avg_query_cosine = similarity_comparison_df["query_cosine_similarity"].mean()

# Log scalar values
logger.report_scalar(
    title="Average Cosine Similarity - all-MiniLM-L6-v2",
    series="Embedding Evaluation - Retrieval Cosine Similarity",
    value=avg_retrieval_cosine,
    iteration=0
)

logger.report_scalar(
    title="Average Cosine Similarity - all-MiniLM-L6-v2",
    series="Embedding Evaluation - Query Cosine Similarity",
    value=avg_query_cosine,
    iteration=0
)

In [ ]:
path = "evaluation_data/retrieval_comparison.csv"
comparison_df.to_csv(path)